# Классификация обращений

Научим модель выбирать отдел по сообщению студента. Сравним два способа представить текст — слова и символьные фрагменты — и посмотрим, какие обращения вызывают ошибки.

Начнём с простой точки отсчёта: насколько полезно всегда отвечать одной категорией? Затем обучим классификатор и сравним его результат с этим правилом.

In [ ]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Данные и сравнение

Используем корпус обращений из первой работы. Готовые части train, validation и test находятся в столбце `split`; `group_id` связывает перефразировки одной ситуации.

Модели по словам и символам будут обучаться на одном train. Для выбора используем macro-F1 на validation; при равенстве оставляем вариант words. К test перейдём после выбора. На этом наборе уже не будем подбирать настройки.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

df = pd.read_csv(DATA / "requests.csv", keep_default_na=False)
train = df[df["split"].eq("train")].copy()
valid = df[df["split"].eq("validation")].copy()
test = df[df["split"].eq("test")].copy()
labels = sorted(df["label"].unique())
print({k: len(v) for k, v in {"train": train, "validation": valid, "test": test}.items()})

### Как читать предсказания

В маленьком примере ниже известны настоящие категории и три ответа воображаемого классификатора. Найдите ошибку до запуска. Какую долю сообщений он направил правильно?

In [ ]:
demo_true = np.array(["it", "library", "library"])
demo_pred = np.array(["it", "it", "library"])
display(pd.DataFrame({"true":demo_true, "predicted":demo_pred,
                      "correct":demo_true == demo_pred}))
print("Доля верных ответов:", np.mean(demo_true == demo_pred))

Правильны два ответа из трёх. Ошибочное библиотечное обращение попало в ИТ. Дальше будем получать такие предсказания от обученной модели и смотреть не только на общий процент, но и на ошибки по категориям.

## 1. Разделяем ситуации, а не перефразировки

Проверьте, что одна группа не встречается сразу в двух частях корпуса. Сохраните множества `group_id` для train, validation и test в словарь `groups`, затем найдите пересечения каждой пары.

Посмотрите на число примеров каждой категории в каждой части. Все ли шесть отделов представлены? Почему пустого пересечения идентификаторов строк было бы недостаточно для проверки этого корпуса?

In [ ]:
groups = {}
# Ваш код
raise NotImplementedError

In [ ]:
assert set(groups) == {"train", "validation", "test"}
assert len(set.union(*groups.values())) == df["group_id"].nunique()

## 2. От трёх документов к TF-IDF

Для начала возьмём три коротких текста из лекции. Обучите на них `toy_vectorizer`, сохраните матрицу в `toy_matrix` и выведите словарь, IDF и матрицу. Маленький результат удобно показать через `toarray()`, чтобы видеть каждую координату.

Проверьте столбец «книга»: слово дважды встретилось в первой строке, но df по всему набору равно 2. Объясните различие этих счётчиков. Сопоставьте первый нормированный вектор с ручным расчётом.

Большая матрица основного корпуса останется разреженной; её целиком переводить в `toarray()` не нужно.

In [ ]:
toy = ["книга книга библиотека", "книга продление", "общежитие оплата"]
# Ваш код
raise NotImplementedError

In [ ]:
assert toy_matrix.shape == (3, 5)
assert np.allclose(np.asarray(toy_matrix.multiply(toy_matrix).sum(axis=1)).ravel(), 1)
book_index = toy_vectorizer.vocabulary_["книга"]
assert np.isclose(toy_vectorizer.idf_[book_index], np.log(4 / 3) + 1)

## 3. Постоянный ответ как baseline

Обучите на train `DummyClassifier(strategy="most_frequent")`. Он будет возвращать одну категорию независимо от сообщения. Проверьте его на validation.

Напишите `score_predictions`, возвращающую словарь с ключами `accuracy` и `macro_f1`. Для F1 передавайте весь список `labels` и `zero_division=0`, чтобы одинаково учитывать все категории даже при отсутствии предсказаний некоторых из них. Результат baseline сохраните в `baseline_scores`.

Сколько правильных ответов можно было ожидать по распределению классов? Почему macro-F1 получилась заметно ниже accuracy?

In [ ]:
def score_predictions(y_true, y_pred):
    # Ваш код
    raise NotImplementedError

# Ваш код
raise NotImplementedError

In [ ]:
assert 0 <= baseline_scores["accuracy"] <= 1
assert 0 <= baseline_scores["macro_f1"] <= 1

## 4. Слова и символы

Постройте два Pipeline с одной и той же логистической регрессией. В words используйте `TfidfVectorizer(ngram_range=(1,2), max_features=5000)`. В chars — `TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=5000)`. Классификатор в обоих случаях: `LogisticRegression(C=1, max_iter=1000, random_state=42)`.

Обучите оба варианта на train и сравните их на validation. Сохраните конвейеры в `models`, оценки в `validation_scores`, имя выбранного варианта в `selected_name`, сам конвейер в `selected`. Основной критерий — macro-F1; при равенстве выбираем words.

Какой тип различий между обращениями может лучше сохранять символьное представление? Приведите пару словоформ или опечатку в качестве примера.

In [ ]:
models = {}
validation_scores = {}
# Ваш код
raise NotImplementedError

In [ ]:
assert set(models) == {"words", "chars"}
assert selected_name in models
assert selected is models[selected_name]
assert all(0 <= s["macro_f1"] <= 1 for s in validation_scores.values())

## 5. Где модель ошиблась

Примените выбранный конвейер к test, сохраните предсказания в `pred_test`, а метрики — в `test_scores`. Постройте матрицу ошибок: настоящие классы по строкам, предсказанные по столбцам.

Создайте таблицу `errors` с ошибочными обращениями. Выберите до пяти разных `group_id` для подробного разбора. Если ошибок меньше, разберите доступные и добавьте отдельные контрольные сообщения, не меняя модель.

Сравните несколько вариантов одной ошибочной группы. Ошибка сохраняется при перефразировании или появляется только в части строк? Проверьте также три контрольных входа: пустую строку, «жжжжжжжжж» и «Не открывается виртуальная комната конференции». Что определяет ответ, когда текст не даёт ни одного известного признака?

In [ ]:
# Ваш код
raise NotImplementedError

In [ ]:
assert len(pred_test) == len(test)
assert set(pred_test) <= set(labels)
assert len(errors) == int(np.sum(test["label"].to_numpy() != pred_test))

## Разбор результата

Укажите выбранное представление и сопоставьте validation с test. Прочитайте ошибки из нескольких разных групп: какие формулировки оказались трудными?

Выберите одно ошибочное обращение и предложите объяснение. Что нужно дополнительно посмотреть или изменить, чтобы проверить эту гипотезу? Отдельно объясните, что произошло бы при случайном разделении стилистических вариантов одной группы между train и test.

Сдайте выполненный ноутбук с таблицами, матрицей ошибок и своими пояснениями. Отдельный отчёт по тем же результатам не нужен.